# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [28]:
import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown,display,update_display
from openai import OpenAI
import ollama

In [29]:
load_dotenv(override=True)
api_key=os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.strip().startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [30]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_PHI = 'phi3'

In [31]:
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [32]:
# set up environment
openai = OpenAI()

In [45]:
system_prompt = """You are **Utv**, an intelligent and friendly online
 code tutor and programming assistant.Your job is to analyze, explain, debug, refactor, and optimize code across multiple programming languages.
  You help users understand how code 
 works, identify bugs, improve performance, and follow best practices.
You must always respond **only in Markdown format**, using clear headings, bullet points, and fenced code
 blocks where appropriate. Explanations should be simple, structured, and beginner-friendly while remaining technically accurate. When debugging or optimizing, explain the 
 reasoning behind changes. If a user’s question or code is unclear, ask concise clarifying questions before answering.
"""
messages=[
    {"role":"system","content":system_prompt},
    {"role":"user","content":question}
]

In [46]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""


In [48]:
stream=openai.chat.completions.create(model=MODEL_GPT,messages=messages,stream=True)
response = ""
display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    response += chunk.choices[0].delta.content or ''
    response = response.replace("```","").replace("markdown", "")
    update_display(Markdown(response), display_id=display_handle.display_id)

## Explanation of the Code

The code snippet you provided uses a combination of a generator with the `yield from` statement and a set comprehension. Let's break it down step by step:

### Components of the Code

1. **Set Comprehension**: 
   python
   {book.get("author") for book in books if book.get("author")}
   
   - **Purpose**: Creates a set of authors from a collection of `books`.
   - **Iteration**: It iterates over each `book` in the `books` collection.
   - **Condition**: It checks if the `author` exists using `book.get("author")`. If the author is present (not `None` or an empty string), it adds the `author` to the set.
   - **Result**: The result of this comprehension is a set that contains only unique authors, since sets automatically discard duplicates.

2. **Yield from**:
   python
   yield from ...
   
   - **Purpose**: This is a generator statement in Python that allows a generator to yield all the values from another iterable (in this case, the set created above).
   - **Functionality**: Instead of yielding each value manually, `yield from` efficiently forwards the yielded values one by one to the caller.

### What the Code Does

- The complete expression `yield from {book.get("author") for book in books if book.get("author")}` creates a generator that iterates over unique authors extracted from the `books` list.
  
- When called, this code will yield each author who is present in the list of books, ensuring that each author is yielded only once (thanks to the properties of a set).

### Why It Is Useful

- **Efficiency**: Using a set comprehension ensures that the list of authors is unique and removes any duplicate entries automatically.
- **Simplicity**: The code is concise and leverages Python's built-in features (like comprehensions) to create readable and efficient code.

### Example

Consider the following example `books` list:

python
books = [
    {"title": "Book A", "author": "Author 1"},
    {"title": "Book B", "author": "Author 2"},
    {"title": "Book C", "author": "Author 1"},
    {"title": "Book D", "author": None},
]


Using the provided code would yield:


Author 1
Author 2


### Summary

- **Functionality**: Yields unique authors from a list of books.
- **Method**: Combines set comprehension and `yield from` for clean and efficient code.

Feel free to ask if you have any further questions or need more clarifications!

OLLAMA_API="http://localhost:11434/api/chat"
HEADERS={"Content-Type":"application/json"}


In [ ]:
!ollama pull phi3

In [37]:
# Get gpt-4o-mini to answer, with streaming
stream = ollama.chat(model=MODEL_PHI,messages=messages,stream=True)
response=""
display_handle=display(Markdown(""),display_id=True)
for chunk in stream:
    response+=chunk.get("message",{}).get("content","")
    response=response.replace("```","").replace("markdown","")
    update_display(Markdown(response),display_id=display_handle.display_id)
    


# Explanation of the Code Snippet `yield from {book.get("author") for book in books if book.get("author")}`

This Python code snippet uses a generator expression within an iterator (though it is not strictly correct syntax as per PEP8) to extract authors' names from a collection called `books`. Here’s what the line of code does, broken down step by step:

1. The `{book.get("author") for book in books if book.get("author")}` part appears at first glance to be an incorrect use of Python syntax; specifically, it seems like a combination between `yield from` (which is used within generator functions) and the comprehension form `(...)` which isn't directly valid or standard outside such contexts in modern Python versions. However, I will interpret this as intending to demonstrate how one might chain multiple iterator transformations together with yields using proper syntax:
    - `book.get("author")` attempts to retrieve the value associated with the "author" key from a book dictionary-like object (`book`), which seems like it should return either None (if not present) or some author name as per typical dict behavior in Python 3+ when accessing non-existent keys using `.get()`.
    - `for book in books if book.get("author")` is intended to iterate over the collection of `books`, but includes a condition that only considers those where "author" exists (i.e., does not return None). As written, this would filter out empty or missing entries before attempting retrieval—but note as above it's mixing comprehension and generator patterns in an unconventional way which won’t work directly without proper function wrapping:
    - The `{...}` part is a set literal in Python that evaluates to the same elements present within, removing duplicates; however here we intend for `yield from` usage. We'll correct this by moving it inside of an appropriate generator context as further explained below.

2. To fix and make sense out of what appears to be intended code while using best practices in Python:
    - Replace `{...}` with a proper list comprehension, because the intention seems to create a set from authors' names without duplicates (since each book might have one author). Lists don't remove duplicates by default unless explicitly handled. The `.get()` method won’t work within this context as-is since it doesn't return None but rather 0 or an empty string, which is not suitable for distinguishing non-existent "author" keys:
    - Replace `yield from` with a proper function that returns the values produced by the comprehension and yields them. In Python versions before 3.3 (where generator delegation was introduced), one would typically use this technique within an iterative context, but here we'll write it as standalone:
    - Finally, wrap all of these in a simple function call that will execute our intended operation using the standard library `itertools` module for better-suited functionality. Since Python 3.x allows yield from without requiring generator functions and generators are just iterable objects themselves when used directly with iteration tools:

Here's what you want to write as valid code, which accomplishes a similar goal of collecting unique authors’ names safely by handling non-existent keys gracefully using `.get()` while avoiding inefficiencies and bugs. This approach uses list comprehension for clarity but then flattens the results into a set:

python
from itertools import chain

# Assuming 'books' is your collection of book dictionaries like books = [{'author': 'Jane Doe'}, {'title': ...}, etc.]
def get_unique_authors(books):
    authors = (book.get('author') for book in books if ('author',) in book and len(book['author']) > 0) # Filter out non-existing or empty 'author' entries explicitly to avoid None issues with .get() when it’s not necessary, as per the code logic
    return set(authors)   # Convert list of unique authors into a set for efficiency and distinctiveness. In Python >=3.7 dictionaries preserve insertion order which means if you need ordered output too (earlier inserts come before later ones), then using OrderedDict from collections module might be better, but here we'll just go with the regular dict as this is more common scenario

Now to call and use our function:

python
unique_authors = get_unique_authors(books)
for author in unique_authors:
    print(author)  # Or process each author name further if needed.

Note that this refactor addresses the concerns with using `.get()` to handle non-existent keys and also ensures we are collecting only authors' names (as opposed to potential empty strings or None).

In [ ]:
# Get Llama 3.2 to answer